# 03 Analysis

Segunda fase del flujo. Este notebook parte desde las salidas *_preprocessed_long.csv de **preprocesamiento** (compiladas via `ttl.load_all_preprocessed_long`) y no recalcula normalizacion, `ROI_status`, `phase`, `trend` ni las metricas `low/mid/high` ya generadas.

Entrada canonica de esta fase: todas las carpetas bajo `data/Proc_data/*/  *_preprocessed_long.csv`, filtradas por `ROI_status == 1`.

Primer objetivo: filtrar por `trend` y visualizar una curva fina de `NormSignal` por temperatura.

In [1]:
%matplotlib qt
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(str(Path("../scripts").resolve()))
import analisis_ttl as ttl

base_dir = Path("/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data")

## Parametros

Edita estos valores para explorar otros subconjuntos sin cambiar el procesamiento original.

In [2]:
trend_filter = ["increase"]
mutant_filter = ["m27", "m36", "m45", "m65"]  # None conserva todos; ejemplo: ["m27", "m36"]
phase_filter = "cooling"
temp_range = (25, 40)  # None conserva todo el rango; ejemplo: (20, 40)
bin_width = 1.0
value_col = "NormSignal"

# Usa [] para una curva global, o columnas como ["genotype_meta"] o ["genotype_meta", "sample"].
curve_group_cols = ["genotype_meta"]

In [3]:
imports = ttl.load_all_preprocessed_long(base_dir)

load_status = imports["load_status"]
preprocessed_all = imports["preprocessed_all"]

print(f"preprocessed_all: {preprocessed_all.shape}")
if preprocessed_all.empty:
    print("No se encontraron archivos *_preprocessed_long.csv. Ejecuta primero 01_preprocessing.ipynb para cada muestra.")

# Cada carpeta debe tener un unico *_preprocessed_long.csv. Mas de uno
# generalmente significa que quedo un CSV viejo sin borrar de una corrida
# anterior de 01_preprocessing.ipynb (ver notas de esa celda de exportacion),
# y se estarian duplicando ROIs de esa muestra en preprocessed_all.
duplicated_preprocessed = load_status[load_status["n_files"] > 1]
if not duplicated_preprocessed.empty:
    print("ADVERTENCIA: carpetas con mas de un *_preprocessed_long.csv (revisar y dejar solo el correcto):")
    print(duplicated_preprocessed[["folder", "preprocessed_files"]].to_string(index=False))

load_status

preprocessed_all: (236630, 43)


,folder,preprocessed_files,n_files,status
0,mut27_image10,[sample_10_m27_cooling_preprocessed_long.csv],1,ok
1,mut27_image11,[sample_11_m27_cooling_preprocessed_long.csv],1,ok
2,mut36_image8,[sample_08_m36_cooling_preprocessed_long.csv],1,ok
3,mut45_image13,[sample_13_m45_cooling_preprocessed_long.csv],1,ok
4,mut57_image2,[sample_02_m57_cooling_preprocessed_long.csv],1,ok
5,mut57_image3,[sample_03_m57_cooling_preprocessed_long.csv],1,ok
6,mut57_image4,[sample_04_m57_cooling_preprocessed_long.csv],1,ok
7,mut65_image19,[sample_19_m65_cooling_preprocessed_long.csv],1,ok


## Cargar salida de procesamiento

Esta fase parte de `preprocessed_all` (compilado de todas las carpetas via `ttl.load_all_preprocessed_long`, celda de arriba) y se queda solo con las ROI marcadas como incluidas (`ROI_status == 1`). Si falta alguna columna minima, el notebook se detiene para evitar analisis ambiguos.

In [4]:
required_cols = {
    "sample",
    "genotype_meta",
    "ROI",
    "trend",
    "phase",
    "temp_mean",
    value_col,
}

if preprocessed_all.empty:
    raise ValueError("preprocessed_all esta vacio. Revisa la celda de carga de _preprocessed_long.csv mas arriba.")

processing_active = preprocessed_all[preprocessed_all["ROI_status"] == 1].copy()
processing_active = processing_active.rename(columns={"genotype": "genotype_meta"})

missing_cols = required_cols - set(processing_active.columns)
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas en preprocessed_all: {sorted(missing_cols)}")

print("Fuente: preprocessed_all (ROI_status == 1)")
print(f"processing_active: {processing_active.shape}")
print(f"ROIs: {processing_active['ROI'].nunique()}")
print(f"Muestras: {processing_active['sample'].nunique()}")
print(f"Genotipos: {processing_active['genotype_meta'].nunique()}")

Fuente: preprocessed_all (ROI_status == 1)
processing_active: (70607, 43)
ROIs: 221
Muestras: 8
Genotipos: 5


## Filtrar por fase y trend

El filtro respeta las etiquetas heredadas desde procesamiento; no las recalcula.

In [5]:
analysis_df = processing_active.copy()

if phase_filter is not None:
    analysis_df = analysis_df[analysis_df["phase"].astype(str).eq(str(phase_filter))].copy()

if trend_filter:
    selected_trends = {str(trend) for trend in trend_filter}
    analysis_df = analysis_df[analysis_df["trend"].astype(str).isin(selected_trends)].copy()

if mutant_filter:
    selected_mutants = {str(mutant) for mutant in mutant_filter}
    analysis_df = analysis_df[analysis_df["genotype_meta"].astype(str).isin(selected_mutants)].copy()

analysis_df["temp_mean"] = pd.to_numeric(analysis_df["temp_mean"], errors="coerce")
analysis_df[value_col] = pd.to_numeric(analysis_df[value_col], errors="coerce")
analysis_df = analysis_df.dropna(subset=["temp_mean", value_col, "ROI"]).copy()

if temp_range is not None:
    temp_min, temp_max = temp_range
    analysis_df = analysis_df[
        analysis_df["temp_mean"].between(temp_min, temp_max, inclusive="both")
    ].copy()

if analysis_df.empty:
    raise ValueError("El filtro no dejo filas para analizar. Revisa phase_filter, trend_filter, mutant_filter y temp_range.")

print(f"phase_filter: {phase_filter}")
print(f"trend_filter: {trend_filter}")
print(f"mutant_filter: {mutant_filter}")
print(f"temp_range: {temp_range}")
print(f"Filas filtradas: {analysis_df.shape}")
print(f"ROIs filtradas: {analysis_df['ROI'].nunique()}")
print(f"Muestras filtradas: {analysis_df['sample'].nunique()}")
print(f"Genotipos filtrados: {analysis_df['genotype_meta'].nunique()}")

display(
    analysis_df[["sample", "genotype_meta", "ROI", "trend", "phase", "temp_mean", value_col]]
    .head()
)

phase_filter: cooling
trend_filter: ['increase']
mutant_filter: ['m27', 'm36', 'm45', 'm65']
temp_range: (25, 40)
Filas filtradas: (9037, 43)
ROIs filtradas: 152
Muestras filtradas: 5
Genotipos filtrados: 4


,sample,genotype_meta,ROI,trend,phase,temp_mean,NormSignal
2354,sample_10,m27,ROI23,increase,cooling,34.341394,0.028415
2355,sample_10,m27,ROI23,increase,cooling,39.501783,0.035463
2356,sample_10,m27,ROI23,increase,cooling,38.864926,0.047276
2357,sample_10,m27,ROI23,increase,cooling,38.143204,0.033527
2358,sample_10,m27,ROI23,increase,cooling,37.628539,0.041893


## Curva fina por temperatura

Primero se promedia cada ROI dentro de cada bin de temperatura. Luego se resume la distribucion de ROIs por bin para graficar la curva.

In [6]:
temp_start = np.floor(analysis_df["temp_mean"].min() / bin_width) * bin_width
temp_stop = np.ceil(analysis_df["temp_mean"].max() / bin_width) * bin_width + bin_width
temp_bins = np.arange(temp_start, temp_stop + bin_width / 10, bin_width)

binned_df = analysis_df.copy()
binned_df["temp_bin"] = pd.cut(
    binned_df["temp_mean"],
    bins=temp_bins,
    include_lowest=True,
    right=False,
)

roi_group_cols = ["genotype_meta", "sample", "ROI", "temp_bin"]
roi_bin_summary = (
    binned_df.dropna(subset=["temp_bin"])
    .groupby(roi_group_cols, observed=True, dropna=False)
    .agg(
        temp_center=("temp_mean", "mean"),
        signal_mean=(value_col, "mean"),
        n_points=(value_col, "size"),
    )
    .reset_index()
)

summary_group_cols = [col for col in curve_group_cols if col in roi_bin_summary.columns]
curve_summary = (
    roi_bin_summary
    .groupby(summary_group_cols + ["temp_bin"], observed=True, dropna=False)
    .agg(
        temp_center=("temp_center", "mean"),
        mean_signal=("signal_mean", "mean"),
        sd_signal=("signal_mean", "std"),
        n_roi=("ROI", "nunique"),
    )
    .reset_index()
)
curve_summary["sem_signal"] = curve_summary["sd_signal"] / np.sqrt(curve_summary["n_roi"])

if summary_group_cols:
    phenotype_counts = (
        analysis_df
        .groupby(summary_group_cols, dropna=False)
        .agg(
            n_roi=("ROI", "nunique"),
            n_samples=("sample", "nunique"),
            n_rows=(value_col, "size"),
        )
        .reset_index()
    )
else:
    phenotype_counts = pd.DataFrame({
        "phenotype": ["all"],
        "n_roi": [analysis_df["ROI"].nunique()],
        "n_samples": [analysis_df["sample"].nunique()],
        "n_rows": [len(analysis_df)],
    })

print(f"bin_width: {bin_width} °C")
print(f"ROI x bin: {roi_bin_summary.shape}")
print(f"Curva resumida: {curve_summary.shape}")
print("n por fenotipo presentado en el grafico:")
display(phenotype_counts)

display(curve_summary.head())

bin_width: 1.0 °C
ROI x bin: (2895, 7)
Curva resumida: (60, 7)
n por fenotipo presentado en el grafico:


,genotype_meta,n_roi,n_samples,n_rows
0,m27,91,2,4716
1,m36,56,1,3192
2,m45,13,1,689
3,m65,10,1,440


,genotype_meta,temp_bin,temp_center,mean_signal,sd_signal,n_roi,sem_signal
0,m27,"[25.0, 26.0)",25.447803,0.001460,0.007863,91,0.000824
1,m27,"[26.0, 27.0)",26.552046,0.006501,0.019742,91,0.002070
2,m27,"[27.0, 28.0)",27.485034,0.014157,0.029913,91,0.003136
3,m27,"[28.0, 29.0)",28.515161,0.026083,0.045906,91,0.004812
4,m27,"[29.0, 30.0)",29.545710,0.042698,0.061475,91,0.006444


In [7]:
group_col = "genotype_meta"
genotypes = sorted(analysis_df[group_col].dropna().astype(str).unique())

ncols = 2
nrows = int(np.ceil(len(genotypes) / ncols)) if genotypes else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharex=True, sharey=True, squeeze=False)
axes_flat = axes.flatten()

for ax, genotype in zip(axes_flat, genotypes):
    roi_sub = roi_bin_summary[roi_bin_summary[group_col].astype(str).eq(genotype)]
    for _, roi_curve in roi_sub.groupby(["sample", "ROI"], observed=True):
        roi_curve = roi_curve.sort_values("temp_center")
        ax.plot(roi_curve["temp_center"], roi_curve["signal_mean"], color="grey", alpha=0.25, linewidth=0.8)

    mean_sub = curve_summary[curve_summary[group_col].astype(str).eq(genotype)].sort_values("temp_center")
    sem = mean_sub["sem_signal"].fillna(0)
    ax.plot(mean_sub["temp_center"], mean_sub["mean_signal"], color="crimson", linewidth=2.2, marker="o", markersize=4, label="promedio")
    ax.fill_between(mean_sub["temp_center"], mean_sub["mean_signal"] - sem, mean_sub["mean_signal"] + sem, color="crimson", alpha=0.2)

    n_roi = roi_sub[["sample", "ROI"]].drop_duplicates().shape[0]
    ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.4)
    ax.set_title(f"{genotype} (n_roi={n_roi})")
    ax.set_xlabel("Temperature (°C)")
    ax.legend(loc="upper left", fontsize=8)

for ax in axes_flat[: len(genotypes)]:
    if ax.get_subplotspec().is_first_col():
        ax.set_ylabel(value_col)
for ax in axes_flat[len(genotypes):]:
    ax.axis("off")

fig.suptitle(f"Curvas por ROI + promedio | trend={trend_filter} | phase={phase_filter} | temp={temp_range}")
fig.tight_layout()
plt.show()